# 🇪🇺 Council of Europe (eProc) — Fetch & Upload

https://eproc.coe.int/callfortenders-list

Queries the Council of Europe's tender portal for all currently live, published call-for-tenders notices, and publishes new ones to the OppsLink job board.

**Filters applied: none, deliberately.**

Javiera's read after reviewing the site: there are relatively few live opportunities here and most look at least somewhat relevant, so this source is uploaded unfiltered by CPV or category rather than adding filtering complexity for little benefit (unlike the other five sources).

**Investigation notes (context for anyone maintaining this later):**
- This site does NOT use CPV as its primary taxonomy. It has its own internal `categoriesCoe` scheme (e.g. `0100` "Functional Or Technical Services"), and only about half of live tenders even have a CPV code attached. Out of the 29-code CPV list used elsewhere in this project, filtering by CPV here only ever matched 1 of 25 live tenders — confirmed via live testing, not assumption.
- No public API docs exist. Endpoint, params and response shape below were all confirmed via live Network-tab capture against `eproc.coe.int/api/CallForTenders/filtered-public-paginated`, same reverse-engineering approach as UNGM and EU Commission.
- `nbTotalElements` in the API response is the all-time, all-status grand total (drafts, closed, cancelled, everything) — NOT the live count. `nbTotalFilteredElements` is the one that respects the `status=Published` filter and matches what's actually visible on the site.
- The API returns HTTP `204 No Content` (not `200` with an empty list) when zero tenders match a filter. Handled explicitly below — an unhandled 204 will raise a `JSONDecodeError` on `.json()`.

**⚠️ Known gaps — confirm before running in production:**
1. **No `description` field exists in this endpoint's response** (confirmed by inspecting the raw JSON — fields are `title`, `status`, `deadline`, `publishedAt`, `categoriesCoe`, `categoriesCpv`, `countries`, `areas`, `id`, timestamps; nothing else). There may be a separate detail-page endpoint (like UNGM's per-notice GET) that has one — not yet checked. Until confirmed, the `description` posted to OppsLink is a placeholder built from the title + the tender's own CoE/CPV category labels, not real free-text description content.
2. **No confirmed individual tender permalink field.** The `link` value below is an unverified guess at the URL pattern (`https://eproc.coe.int/callfortenders/{id}`) based on the list page's own URL — not captured from a real click-through. Confirm the real pattern (open one live tender in the browser, copy the URL) before this goes live.
3. **No confirmed reference/notice-number field**, so "Call Identifier" is posted as "Not specified" for every contract from this source.
4. **Employer is hardcoded to "Council of Europe"** — this is a single-institution portal with no separate contracting-authority field in the list response, unlike EU Commission/TED which have multiple buyers. Flagging as an assumption, not a confirmed absence of a buyer field.
5. **"CoE Categories" is a NEW custom field** (same situation as UNGM's "UNSPSC Codes") — confirm it exists in the Smart Job Board admin panel before running in production, or swap to an existing field.

### Classification Logic

In [7]:
import sys
from pathlib import Path
import json
from copy import deepcopy
from typing import Optional, Tuple

CLASSIFIER_PATH = Path("../classification_test.py")  # adjust if needed
if not CLASSIFIER_PATH.exists():
    raise FileNotFoundError(f"Could not find {CLASSIFIER_PATH.resolve()}")

sys.path.insert(0, str(CLASSIFIER_PATH.parent.resolve()))

from classification_test import classify_expertise, parse_cpv_codes, EXPERTISE, LABEL_DELIM

ALLOWED_EXPERTISE = set(EXPERTISE)

def get_expertise_labels(pred: dict, include_other: bool = True) -> list[str]:
    """
    Returns the list of expertise labels to set on the multi-select field.
    Uses the classifier's multi-label output (primary + secondary).
    """
    labels = []
    try:
        labels = json.loads(pred.get("expertise_pred_all_json", "[]"))
    except Exception:
        s = (pred.get("expertise_pred_all", "") or "").strip()
        labels = [x.strip() for x in s.split(LABEL_DELIM) if x.strip()] if s else []

    labels = [x for x in labels if x in ALLOWED_EXPERTISE]

    if not include_other:
        labels = [x for x in labels if x != "Other"]

    if not labels:
        labels = ["Other"] if include_other else []

    return labels


def predict_expertise_for_contract(title: str, additional_fields: dict) -> tuple[dict, list[str]]:
    desc = (additional_fields.get("description") or "").strip()
    # Council of Europe tenders don't all carry a CPV code (only ~half do) -
    # parse_cpv_codes handles an empty string fine and just returns [], so the
    # classifier falls back to title/description keyword matching for those.
    cpv_list = parse_cpv_codes(additional_fields.get("cpv_codes"))
    pred = classify_expertise(title=title, description=desc, cpv_codes=cpv_list)
    labels = get_expertise_labels(pred, include_other=True)
    return pred, labels

### Get set of jobs already on OppsLink

In [8]:
import requests

def get_all_job_titles(api_key, base_url="https://opps-link.com/api/jobs"):
    """Retrieve ALL active job titles from the API by paginating through all results."""
    job_titles = set()
    page = 1
    limit = 100
    total_jobs = 0

    print("Starting to collect all job titles...")

    while True:
        params = {'page': page, 'limit': limit, 'api_key': api_key}
        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            data = response.json()

            current_jobs = data.get('jobs', [])
            if not current_jobs:
                break

            for job in current_jobs:
                job_titles.add(job['title'])

            total_jobs += len(current_jobs)
            print(f"Processed page {page} - found {len(current_jobs)} jobs (total: {total_jobs})")

            if len(current_jobs) < limit:
                break
            page += 1

        except requests.exceptions.RequestException as e:
            print(f"Error on page {page}: {e}")
            break

    print(f"\nFinished! Collected {len(job_titles)} unique titles from {total_jobs} jobs across {page} pages")
    return job_titles

API_KEY_OPPSLINK = "37c597e7bb52d26099ede8b8aa43b270"
all_titles = get_all_job_titles(API_KEY_OPPSLINK)
all_titles = set(all_titles)

Starting to collect all job titles...
Processed page 1 - found 100 jobs (total: 100)
Processed page 2 - found 100 jobs (total: 200)
Processed page 3 - found 100 jobs (total: 300)
Processed page 4 - found 100 jobs (total: 400)
Processed page 5 - found 100 jobs (total: 500)
Processed page 6 - found 100 jobs (total: 600)
Processed page 7 - found 100 jobs (total: 700)
Processed page 8 - found 100 jobs (total: 800)
Processed page 9 - found 100 jobs (total: 900)
Processed page 10 - found 100 jobs (total: 1000)
Processed page 11 - found 100 jobs (total: 1100)
Processed page 12 - found 100 jobs (total: 1200)
Processed page 13 - found 100 jobs (total: 1300)
Processed page 14 - found 100 jobs (total: 1400)
Processed page 15 - found 100 jobs (total: 1500)
Processed page 16 - found 100 jobs (total: 1600)
Processed page 17 - found 100 jobs (total: 1700)
Processed page 18 - found 100 jobs (total: 1800)
Processed page 19 - found 100 jobs (total: 1900)
Processed page 20 - found 100 jobs (total: 2000)


### Get employers

In [9]:
def get_all_employers(api_key, base_url="https://opps-link.com/api/employers"):
    """Retrieve ALL employers with their IDs from the API by paginating through all results."""
    employers_dict = {}
    employers_list = []
    page = 1
    limit = 100
    total_processed = 0

    print("Starting to collect all employers...")

    while True:
        params = {'page': page, 'limit': limit, 'api_key': api_key}
        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            data = response.json()

            current_employers = data.get('employers', [])
            if not current_employers:
                break

            for employer in current_employers:
                employer_id = employer['id']
                employer_name = employer['company_name']
                employers_dict[employer_name.lower()] = employer_id
                employers_list.append(employer)

            total_processed += len(current_employers)
            print(f"Processed page {page} - found {len(current_employers)} employers (total: {total_processed})")

            if len(current_employers) < limit:
                break
            page += 1

        except requests.exceptions.RequestException as e:
            print(f"Error on page {page}: {e}")
            break

    print(f"\nFinished! Collected {len(employers_dict)} unique employers across {page} pages")
    return employers_dict, employers_list

employers_dict, employers_list = get_all_employers(API_KEY_OPPSLINK)

Starting to collect all employers...
Processed page 1 - found 100 employers (total: 100)
Processed page 2 - found 100 employers (total: 200)
Processed page 3 - found 100 employers (total: 300)
Processed page 4 - found 100 employers (total: 400)
Processed page 5 - found 100 employers (total: 500)
Processed page 6 - found 100 employers (total: 600)
Processed page 7 - found 100 employers (total: 700)
Processed page 8 - found 100 employers (total: 800)
Processed page 9 - found 100 employers (total: 900)
Processed page 10 - found 100 employers (total: 1000)
Processed page 11 - found 100 employers (total: 1100)
Processed page 12 - found 100 employers (total: 1200)
Processed page 13 - found 100 employers (total: 1300)
Processed page 14 - found 100 employers (total: 1400)
Processed page 15 - found 100 employers (total: 1500)
Processed page 16 - found 100 employers (total: 1600)
Processed page 17 - found 38 employers (total: 1638)

Finished! Collected 1623 unique employers across 17 pages


### OppsLink helpers — employer creation, job creation, shared utilities

Copied verbatim from `eu_commission.ipynb` / `ungm.ipynb` so this notebook is self-contained and can run standalone, same convention as the other five source notebooks.

In [10]:
from urllib.parse import quote
import re, html
from datetime import datetime, timezone
from dateutil import parser as _dateparser

LOGO_DEV_TOKEN = "pk_Z6-BMh5ES2ukcUOHzySQmw"
JOB_API_URL = "https://opps-link.com/api/jobs"
EMPLOYER_API_URL = "https://opps-link.com/api/employers"
HEADERS = {"Content-Type": "application/json"}

_employer_email_cache = {}  # {employer_id: email}

def _slugify(name: str, max_len: int = 50) -> str:
    s = re.sub(r'[^a-z0-9]+', '-', (name or '').lower()).strip('-')
    return s[:max_len] or 'emp'

def _alias_for(company_name: str) -> str:
    return f"consulting+{_slugify(company_name)}@lse.ac.uk"

def fetch_logo_from_logo_dev_by_name(company_name: str):
    """Fetch logo bytes from Logo.dev using company-name lookup. Returns (filename, content_bytes, mime_type) or None."""
    company_name = (company_name or "").strip()
    if not company_name:
        return None

    encoded_name = quote(company_name)
    logo_url = (
        f"https://img.logo.dev/name/{encoded_name}"
        f"?token={LOGO_DEV_TOKEN}&format=png&size=256&fallback=404"
    )

    try:
        resp = requests.get(logo_url, timeout=30)
        if resp.status_code != 200:
            print(f"⏭️ No logo found for '{company_name}' ({resp.status_code})")
            return None
        mime_type = resp.headers.get("Content-Type", "image/png")
        filename = f"{re.sub(r'[^a-z0-9]+', '_', company_name.lower()).strip('_') or 'logo'}.png"
        print(f"✅ Fetched logo for {company_name}")
        return filename, resp.content, mime_type
    except Exception as e:
        print(f"⚠️ Logo lookup failed for '{company_name}': {e}")
        return None


def _create_employer(company_name: str, website: str = "") -> Tuple[bool, Optional[int], str]:
    company_name = (company_name or "Not Disclosed").strip() or "Not Disclosed"
    clean_name = re.sub(r'[^a-z0-9_]', '', company_name.lower().replace(' ', '_'))
    email = _alias_for(company_name)
    password = f"{clean_name or 'emp'}_123!"

    data = {
        "api_key": API_KEY_OPPSLINK,
        "email": email,
        "password": password,
        "company_name": company_name,
        "company_description": f"<p>{company_name}</p>",
        "active": 1,
        "featured": 0,
        "registration_date": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        "full_name": company_name,
        "location": "United Kingdom",
        "website": website or ""
    }

    files = None
    logo_file = fetch_logo_from_logo_dev_by_name(company_name)
    if logo_file:
        filename, content, mime_type = logo_file
        files = {"logo": (filename, content, mime_type)}
    else:
        print(f"⏭️ Creating employer without logo: {company_name}")

    try:
        if files:
            r = requests.post(EMPLOYER_API_URL, data=data, files=files, timeout=60)
        else:
            r = requests.post(EMPLOYER_API_URL, data=data, timeout=60)

        if r.status_code == 201:
            eid = r.json().get("id")
            print(f"✅ Employer created: {company_name} (ID: {eid}) website={website or '-'} logo={'yes' if files else 'no'}")
            return True, eid, email
        else:
            print(f"❌ Failed to create employer {company_name}: {r.status_code} {r.text}")
            return False, None, ""
    except Exception as e:
        print(f"❌ Exception creating employer {company_name}: {e}")
        return False, None, ""


def _get_employer_email(employer_id: int) -> str:
    try:
        r = requests.get(f"{EMPLOYER_API_URL}/{employer_id}",
                         params={"api_key": API_KEY_OPPSLINK},
                         headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        return (data.get("email") or "").strip()
    except Exception as e:
        print(f"⚠️ Could not fetch email for employer {employer_id}: {e}")
        return ""


def clean_description(description):
    """Clean and format the description text."""
    if not description:
        return "Not Disclosed"
    cleaned = html.unescape(description)
    cleaned = cleaned.replace('\r\n', ' ').replace('\n', ' ')
    return ' '.join(cleaned.split()).strip()


def parse_deadline_iso(deadline_iso: str):
    """Parse ISO-like strings (or similar) into a datetime."""
    return _dateparser.parse(deadline_iso) if deadline_iso else None


def is_expired(deadline_iso: str) -> bool:
    try:
        dt = parse_deadline_iso(deadline_iso)
        if not dt:
            return False
        return dt.date() < datetime.today().date()
    except Exception:
        return False


def create_job(job_data: dict) -> Optional[int]:
    r = requests.post(JOB_API_URL, json=job_data, headers=HEADERS, timeout=30)
    if r.status_code == 201:
        job_id = r.json().get("id")
        print(f"✅ Posted job: {job_data['title']} (ID: {job_id}) | categories={job_data.get('categories')}")
        return job_id
    else:
        print(f"❌ Failed to post job '{job_data.get('title')}': {r.status_code} {r.text}")
        return None

### Council of Europe — Fetch (no filtering, per Javiera) & Prepare

Endpoint, params and response shape confirmed via live Network-tab capture; no public API docs exist for this site. No CPV or category filter applied here, deliberately — see README above.

In [11]:
import time

BASE_URL = "https://eproc.coe.int/api/CallForTenders/filtered-public-paginated"
COE_SITE_URL = "https://eproc.coe.int"

EMPTY_RESULT = {"result": [], "nbTotalElements": None, "nbTotalFilteredElements": 0}

def fetch_coe_page(offset: int, limit: int = 25) -> dict:
    resp = requests.get(BASE_URL, params={
        "offset": offset,
        "limit": limit,
        "status": "Published",
        "language": "EN",
        "orderByParam": "Status",
        "orderByOption": "Ascending",
        # deliberately no categoryCpvId / categoryCoeId param - Javiera wants everything live, unfiltered
    })
    if resp.status_code == 204:
        return EMPTY_RESULT  # zero live tenders - legitimate response, not an error
    resp.raise_for_status()
    return resp.json()


def format_categories(raw_categories) -> tuple[str, str]:
    """
    categoriesCoe / categoriesCpv both come back as a list of dicts with
    'code' and 'description'. Returns (codes_joined, descriptions_joined),
    each de-duplicated and comma-separated, or ("", "") if the list is empty
    (categoriesCpv is empty for roughly half of CoE's live tenders).
    """
    if not raw_categories:
        return "", ""
    codes, descs = [], []
    for cat in raw_categories:
        c = (cat.get("code") or "").strip()
        d = (cat.get("description") or "").strip()
        if c and c not in codes:
            codes.append(c)
        if d and d not in descs:
            descs.append(d)
    return ", ".join(codes), ", ".join(descs)


EFFECTIVE_LIMIT = 25  # confirmed safe via live testing; raising this above 25 has NOT been verified

first_page = fetch_coe_page(offset=0, limit=EFFECTIVE_LIMIT)
total_live = first_page["nbTotalFilteredElements"]
print(f"{total_live} live published tenders on Council of Europe's site right now.")

all_coe_raw = list(first_page["result"])
offset = EFFECTIVE_LIMIT
while offset < total_live:
    page = fetch_coe_page(offset=offset, limit=EFFECTIVE_LIMIT)
    all_coe_raw.extend(page["result"])
    offset += EFFECTIVE_LIMIT
    time.sleep(1)

print(f"✅ Fetched {len(all_coe_raw)} raw tenders (expected {total_live})")
if len(all_coe_raw) != total_live:
    print("⚠️ Mismatch between fetched count and reported total - investigate before uploading.")


# --- Build contract dicts ---
# ⚠️ See README cell at the top for the known gaps this section works around:
# no confirmed `description` field, no confirmed tender permalink, no confirmed
# reference/notice-number field, and the single hardcoded "Council of Europe"
# employer. None of these are silent - each is flagged inline below too.

_warned_about_description_gap = False
_warned_about_link_gap = False

all_coe_contracts = []

for item in all_coe_raw:
    title = (item.get("title") or "").strip()
    if not title:
        continue

    tender_id = item.get("id")
    deadline_iso = item.get("deadline")  # e.g. "2026-08-13T17:00:00+02:00" - dateutil handles this fine

    coe_codes, coe_descs = format_categories(item.get("categoriesCoe"))
    cpv_codes, cpv_descs = format_categories(item.get("categoriesCpv"))

    countries = item.get("countries") or []
    country_names = [c.get("name") for c in countries if c.get("name")]
    location = ", ".join(country_names) if country_names else "Council of Europe (all member states)"

    # No real free-text description field exists on this endpoint (confirmed by
    # inspection - see README). Built from the tender's own category labels as
    # a placeholder, both for the classifier and for the OppsLink posting itself,
    # until a detail-page endpoint (if one exists) is found and confirmed.
    description_parts = [title]
    if coe_descs:
        description_parts.append(f"Council of Europe categories: {coe_descs}.")
    if cpv_descs:
        description_parts.append(f"CPV categories: {cpv_descs}.")
    description = clean_description(". ".join(description_parts))

    if not _warned_about_description_gap:
        print("⚠️ Using title + category labels as a description placeholder for all CoE contracts - "
              "no real free-text description field was found on this endpoint. See README.")
        _warned_about_description_gap = True

    # UNVERIFIED URL pattern - confirm the real one by opening a live tender in
    # the browser and comparing, then fix this in one place.
    link = f"{COE_SITE_URL}/callfortenders/{tender_id}" if tender_id is not None else COE_SITE_URL
    if not _warned_about_link_gap:
        print(f"⚠️ Tender permalink pattern is UNVERIFIED (e.g. {link}) - confirm the real URL before production use.")
        _warned_about_link_gap = True

    all_coe_contracts.append({
        "title": title,
        "deadline": deadline_iso,
        "country": location,
        "client": "Council of Europe",  # single-institution portal - no separate buyer field in this response
        "client_link": COE_SITE_URL,
        "description": description,
        "cpv_codes": cpv_codes,      # "" for tenders with no CPV attached - classifier handles that fine
        "coe_categories": coe_descs,  # "" if somehow uncategorised
        "value": "Not Disclosed",     # no value field on this endpoint
        "link": link,
        "reference": "Not specified",  # no confirmed reference/notice-number field on this endpoint
        "duration": "Not specified",   # no duration field on this endpoint
    })

print(f"✅ {len(all_coe_contracts)} contracts ready for upload (no filtering applied, per Javiera)")

24 live published tenders on Council of Europe's site right now.
✅ Fetched 24 raw tenders (expected 24)
⚠️ Using title + category labels as a description placeholder for all CoE contracts - no real free-text description field was found on this endpoint. See README.
⚠️ Tender permalink pattern is UNVERIFIED (e.g. https://eproc.coe.int/callfortenders/11498) - confirm the real URL before production use.
✅ 24 contracts ready for upload (no filtering applied, per Javiera)


### Upload to OppsLink

In [12]:
# NOTE: "CoE Categories" below is a NEW custom field - unlike "Contract Value",
# "Contract Link", "CPV Codes", "Call Identifier" and "Project Duration", which
# are already used (and presumably already configured in Smart Job Board) by
# the other five sources. Confirm "CoE Categories" exists in the Smart Job
# Board admin panel before running this in production, same situation as
# UNGM's "UNSPSC Codes" field.

for contract in all_coe_contracts:
    title = contract["title"]
    if title in all_titles:
        print(f"⏩ Skipping duplicate title: {title}")
        continue

    client_name = contract["client"]
    client_key = client_name.lower()

    employer_id = employers_dict.get(client_key)
    employer_email = ""

    if employer_id is None:
        print(f"🔍 Employer '{client_name}' not found. Attempting to create...")
        ok, employer_id, employer_email = _create_employer(client_name, website=contract.get("client_link", ""))
        if employer_id:
            employers_dict[client_key] = employer_id
            if employer_email:
                _employer_email_cache[employer_id] = employer_email
        else:
            print(f"🚫 Skipping contract '{title}' — failed to create employer.")
            continue
    else:
        employer_email = _employer_email_cache.get(employer_id) or _get_employer_email(employer_id)
        if employer_email:
            _employer_email_cache[employer_id] = employer_email

    deadline_iso = contract["deadline"]

    if is_expired(deadline_iso):
        print(f"🚫 Skipping expired: {title} (deadline {deadline_iso})")
        continue

    try:
        parsed_deadline = parse_deadline_iso(deadline_iso) if deadline_iso else None
        formatted_deadline = parsed_deadline.date().isoformat() if parsed_deadline else None
    except Exception:
        formatted_deadline = None

    how_to_apply_value = employer_email or "consulting@lse.ac.uk"
    pred, expertise_labels = predict_expertise_for_contract(title, contract)

    job_data = {
        "api_key": API_KEY_OPPSLINK,
        "active": 1,
        "featured": 0,
        "title": title,
        "employer_id": employer_id,
        "description": contract["description"],
        "how_to_apply": how_to_apply_value,
        "activation_date": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        "location": contract["country"],
        "categories": expertise_labels,
        "custom_fields": [
            {"name": "Contract Value", "value": contract["value"]},
            {"name": "Contract Link", "value": contract["link"]},
            {"name": "CPV Codes", "value": contract["cpv_codes"] or "Not Disclosed"},
            {"name": "CoE Categories", "value": contract["coe_categories"] or "Not Disclosed"},
            {"name": "Call Identifier", "value": contract["reference"]},
            {"name": "Project Duration", "value": contract["duration"]},
        ]
    }

    if formatted_deadline:
        job_data["expiration_date"] = formatted_deadline
        job_data["custom_fields"].append(
            {"name": "Application deadline", "value": formatted_deadline, "type": "date"}
        )
    else:
        job_data["custom_fields"].append(
            {"name": "Missing source deadline", "value": "Yes"}
        )

    job_id = create_job(job_data)
    if job_id:
        all_titles.add(title)

print("✅ Council of Europe contracts uploaded to OppsLink")

✅ Posted job: 2026/AO/01 - PROVISION OF CONSULTANCY SERVICES, IN THE FIELD OF NON-DISCRIMINATION, COMBATING HATRED AND RACISM IN THE REPUBLIC OF MOLDOVA (ID: 10641) | categories=['Organisations and Management Consulting', 'Equality, Inclusion and Social Policy']
✅ Posted job: 2026AO21 - Achat chambre froide extérieure -35°C (ID: 10642) | categories=['Other']
✅ Posted job: 2026/AO/19 - CALL FOR TENDERS FOR THE PROVISION OF INTELLECTUAL CONSULTANCY SERVICES WITHIN THE FRAMEWORK OF THE PROGRAMMES OF ACTIVITY OF THE REYKJAVIK PROCESS AND ENVIRONMENT DEPARTMENT (ID: 10643) | categories=['Organisations and Management Consulting', 'Environment and Energy']
✅ Posted job: DGA-26-2672 CONTRAT CADRE TRAVAUX DE PLATRERIE (ID: 10644) | categories=['Housing and Built Environment', 'Urban Planning, Infrastructure and Transport']
✅ Posted job: DGS-26-1030 RENOVATION DES ETANCHEITES PALAIS DE L'EUROPE V2 (ID: 10645) | categories=['Other']
✅ Posted job: DGS-26-3012_Rénovation globale du bâtiment D_Lot 1